(z3-gray-code)=
# 格雷码 ― 构造每次只变化 1 位的数列

## 本章要点

**格雷码**是相邻两个值**只差 1 位**的排列。它用在纠错、编码、电路设计、模拟-数字转换等场合。

- 4 位时，把 0〜15 各排一次，让相邻两个数只差 1 位
- 先用 `Int`（整数）写，再换成 `BitVec` 加速
- 同一个问题，只改变量的类型就可能**快 100 倍以上**

[用 OrTools 求解格雷码](ortools-gray-code)

## 思路

$n$ 位的格雷码，是把 $0$ 到 $2^n-1$ 的数各排一次的数列，满足下面的条件。

- 相邻两个数恰好只差 1 位
- 最后一个和第一个也恰好只差 1 位（能绕一圈回到起点）

用 `Int` 写的时候，需要取出 `x` 的第 `i` 位。用 `x % 2**(i+1) - x % 2**i`，第 `i` 位是 `0` 时得到 `0`，是 `1` 时得到 `2**i`。两个数的这个值不同，就说明这一位不同。

## 用 Int 变量求解

In [1]:
from z3 import *


def gray_code(nbits):
    """返回 nbits 位的格雷码（数的排列）"""
    n = 2 ** nbits
    codes = IntVector('code', n)
    solver = Solver()

    for c in codes:
        solver.add(0 <= c, c < n)
    solver.add(Distinct(codes))

    for c1, c2 in zip(codes, codes[1:] + codes[:1]):
        diff = Abs(c1 - c2)
        # 辅助条件：只差 1 位时，差值是 2 的幂
        solver.add(Or([diff == 2 ** bit for bit in range(nbits)]))
        # 真正的条件：不同的位恰好 1 个
        bit_diff = [c1 % 2**(i + 1) - c1 % 2**i != c2 % 2**(i + 1) - c2 % 2**i
                    for i in range(nbits)]
        solver.add(Sum(bit_diff) == 1)

    solver.add(codes[0] == 0)      # 为了消除旋转，把开头固定为 0

    assert solver.check() == sat
    model = solver.model()
    return [model[c].as_long() for c in codes]


nbits = 4
%time codes = gray_code(nbits)
for c in codes:
    print(f'{c:0{nbits}b}')

CPU times: total: 9.19 s
Wall time: 9.33 s
0000
0001
1001
1011
0011
0111
1111
1110
1010
1000
1100
1101
0101
0100
0110
0010


0 到 15 各排了一次，相邻两个数只差 1 位。

不过用 `Int` 写，Z3 很慢。在这个环境里花了几秒。因为位的操作被写成了数的除法，成了 Z3 不好处理的式子。

## 用 BitVec 加速

处理位的问题，一开始就用位向量类型 `BitVec` 更自然。把 `Int` 换成 `BitVec` 之后，

- 取某一位可以用 `>>` 和 `&` 写
- 两个数哪里不同，用 `^`（XOR）一次就能看出来

`c1 ^ c2` 是「只有不同的位为 1」的数。在格雷码里，这个数会是**2 的幂**（只有 1 个 1 立着）。

```python
diff = c1 ^ c2
solver.add(diff & (diff - 1) == 0, diff != 0)
```

`diff & (diff - 1) == 0` 是判断「`diff` 是否为 2 的幂」的常用技巧。2 的幂减 1，立着的那一位变成 0，它下面的位全变成 1，再和原数做 AND 就得到 0。`diff != 0` 是为了排除「差 0 位（同一个数）」的情况。

In [2]:
def gray_code_bv(nbits):
    """BitVec 版：用位运算解同一个问题"""
    codes = [BitVec(f'code_{i}', nbits) for i in range(2 ** nbits)]
    solver = Solver()
    solver.add(Distinct(codes))

    for c1, c2 in zip(codes, codes[1:] + codes[:1]):
        diff = c1 ^ c2                       # 只有不同的位为 1
        solver.add(diff & (diff - 1) == 0,   # 2 的幂
                   diff != 0)                # 不是 0

    solver.add(codes[0] == 0)

    assert solver.check() == sat
    model = solver.model()
    return [model[c].as_long() for c in codes]


%time codes = gray_code_bv(nbits)
for c in codes:
    print(f'{c:0{nbits}b}')

CPU times: total: 15.6 ms
Wall time: 22.5 ms
0000
1000
1100
0100
0101
0111
0110
1110
1111
1101
1001
1011
1010
0010
0011
0001


这次的 `BitVec` 版一瞬间就结束了。在这个环境里 `Int` 版要 10 秒左右，`BitVec` 版大约 0.03 秒，**相差 300 倍以上**（`Int` 版每次执行要几秒到十几秒，但数量级不会变）。

得到的排列有时和 `Int` 版不同，但两者都是正确的格雷码。因为满足条件的排列不止一种。

## 小结

- 格雷码是「相邻两个只差 1 位」的排列
- 用 `Int` 可以写成「差值是 2 的幂」加「不同的位恰好 1 个」
- 处理位的问题换成 `BitVec` 就很快。用 XOR 一个式子就能写出差异
- `diff & (diff - 1) == 0` 是判断「是否为 2 的幂」的技巧
- 解不止一种。满足条件的排列里，返回的是 Z3 最先找到的那个